# MCP 第 03 课：站在 Client 视角理解协议流

学 MCP 最容易迷糊的一点，是一上来就看 server 装饰器，结果不知道 client 到底在干什么。

所以这节课我们反过来：

> 先从 client 视角看 MCP。


## 一个最重要的心智模型

MCP client 的工作，概括起来就四步：

1. 连接 server
2. 初始化协议会话
3. 列出 server 有哪些能力
4. 调用某个能力

真正跑起来以后，你会发现它并不神秘。


In [ ]:
from pathlib import Path

SERVER_PATH = Path("/Users/siji/Desktop/AI/mcp/src/mcp_course/server.py")
SERVER_PATH.exists()


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def inspect_server():
    server_params = StdioServerParameters(
        command="python",
        args=[str(SERVER_PATH)],
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            prompts = await session.list_prompts()
            return tools, resources, prompts


tools, resources, prompts = await inspect_server()
len(tools.tools), len(resources.resources), len(prompts.prompts)


In [ ]:
[tool.name for tool in tools.tools]


In [ ]:
[resource.uri for resource in resources.resources]


In [ ]:
[prompt.name for prompt in prompts.prompts]


In [ ]:
async def call_one_tool():
    server_params = StdioServerParameters(
        command="python",
        args=[str(SERVER_PATH)],
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(
                "days_between",
                {"start_date": "2026-04-20", "end_date": "2026-05-01"},
            )
            return result


await call_one_tool()


## 这一步真正发生了什么

你刚才虽然只写了几行代码，但背后已经完成了完整的一次协议流程：

- client 启动 server 进程
- 双方建立通信通道
- client 发送初始化请求
- client 请求列出能力
- client 调用其中某个能力
- server 返回结构化结果


## 本课工程直觉

1. **先学 client，再学 server，会更容易理解 MCP 到底在解决什么问题。**
2. **`list_tools()` / `list_resources()` / `list_prompts()` 本质上就是“能力发现”。**
3. **当你能稳定完成 initialize -> list -> call，这条链路就已经通了。**
